# Tool使用的概述

## 1、工具的调用方式

### 1.1 方式1：直接调用

举例：

In [24]:
from langchain_core.tools import tool
#下面这样就创建一个工具了
@tool
def get_weather(city: str) -> str:
   """
   获取指定城市的天气信息
   参数:
   city: 城市名称，如"北京"、"上海"
   返回:
   天气信息字符串
   """
   # 你的实现
   return city + "晴天，温度 15°C"

In [25]:
get_weather.invoke({"city":"北京"})

'北京晴天，温度 15°C'

### 1.2 方式2：基于模型进行调用

举例：

In [14]:
from dotenv import load_dotenv
from langchain.chat_models import init_chat_model
import os

load_dotenv(override=True)

MOONSHOT_API_KEY = os.getenv("MOONSHOT_API_KEY")
MOONSHOT_BASE_URL = os.getenv("MOONSHOT_BASE_URL")

model = init_chat_model(
    model="kimi-k2.6",
    model_provider="openai",
    api_key=MOONSHOT_API_KEY,
    base_url=MOONSHOT_BASE_URL
)

In [17]:
from langchain_core.tools import tool
# 定义工具
@tool
def get_weather(city: str) -> str:
    """获取指定城市的天气"""
    # 你的实现
    return "晴天，温度 15°C"
# 绑定工具
model_with_tools = model.bind_tools([get_weather])
# AI 可以决定是否调用工具
response = model_with_tools.invoke("北京天气如何？")
# response = model_with_tools.invoke("2 + 3 = ？")
# 检查 AI 是否要调用工具
if response.tool_calls:
    print("AI 想调用工具：", response.tool_calls)
else:
    print("AI 直接回答：", response.content)

AI 想调用工具： [{'name': 'get_weather', 'args': {'city': '北京'}, 'id': 'get_weather_0', 'type': 'tool_call'}]


## 2、从Message的流转看工具的调用

举例：

In [18]:
from dotenv import load_dotenv
from langchain.chat_models import init_chat_model
import os

load_dotenv(override=True)

MOONSHOT_API_KEY = os.getenv("MOONSHOT_API_KEY")
MOONSHOT_BASE_URL = os.getenv("MOONSHOT_BASE_URL")

model = init_chat_model(
    model="kimi-k2.6",
    model_provider="openai",
    api_key=MOONSHOT_API_KEY,
    base_url=MOONSHOT_BASE_URL
)

In [28]:
from langchain.messages import HumanMessage
@tool
def get_weather(city: str):
   """获取天气的工具"""
   return f"{city}天气晴朗~"
# 将模型和工具绑定
model_with_tools = model.bind_tools([get_weather])
#以Message的方式来看工具的调用
#首先写了一个Message列表，在这里列表中放了一个HumanMessage，也就是用户的问题
messages = [
    HumanMessage("今天北京天气如何")
]
#然后模型分析了问题之后看有哪些工具是跟问题匹配的，如果有匹配的工具就发出调用工具的请求来使用工具，如果没有匹配的模型就会直接回答
#所以这里模型发现了有能使用的工具，生成调用工具请求response
response = model_with_tools.invoke(messages)
#这个请求response返回的实际是一个AImessage，也就是模型返回的内容
#那么在AImessage中有很多字段，其中也包括调用工具的字段tool_calls
#所以当有匹配的工具时，调用了工具在AImessage中的tool_calls字段里就能看到对应调用工具的信息
#下面这一步就是把调用工具后的信息也放入了message列表中，那么添加过后这个消息列表中就有两条消息了，一条是用户的一条是AI的
messages.append(response)
#因为前面也说了response返回的是一个AImessage，然后AImessage中有很多字段，其中也包括调用工具的字段tool_calls
#所以这一步就是去获取tool_calls字段中的信息，首先response.tool_calls获取到这个列表
tool_calls = response.tool_calls
# #拿到列表后通过for循环去获取列表中的每一个信息
for tool_call in tool_calls:

# #通过for循环获取到每一个信息之后，再用if判断，判断列表中的name这个信息是不是叫get_weather
 if tool_call["name"] == "get_weather":
#  #判断如果是get_weather，那么这里就需要我们主动去让工具调用（因为大模型只能根据问题分析出要调用工具，但不能主动调用）
#  #所以这里要手动通过get_weather工具去主动调用一下tool_call并获取tool_call中的参数
  tool_response = get_weather.invoke(tool_call)
#  #手动通过get_weather获取到tool_call中的参数后，它返回的是ToolMessage类型消息
#  #然后我们把这个ToolMessage类型的消息打印出来
  print(type(tool_response))
#  #打印出来后也把这个toolMessage类型的消息添加到messages列表中，现在message列表中就有了三个消息，分别是用户的、AI的、工具调用的
  messages.append(tool_response)
print("=====================> messages <=====================")
for msg in messages: #通过for循环去获取消息列表中的每条消息然后把每条消息的内容都打印出来
   msg.pretty_print()
print("=====================> messages <=====================")
final_response = model_with_tools.invoke(messages)  #把消息列表最后给到大模型，这个消息列表中已经有了三条消息，分别是用户的问题、AI根据问题检测到有能调用的工具返回的请求、从这个返回的请求中获取到了tool_calls，然后把这个tool_call的信息让工具主动调用一下得到一个tool_message，最后大模型根据这三个信息综合出最终回答
print(f"final_response: \n{final_response}")

<class 'langchain_core.messages.tool.ToolMessage'>
=====================> messages <=====================
================================ Human Message =================================

今天北京天气如何
================================== Ai Message ==================================

我来帮您查询今天北京的天气情况。
Tool Calls:
  get_weather (get_weather_0)
 Call ID: get_weather_0
  Args:
    city: 北京
================================= Tool Message =================================
Name: get_weather

北京天气晴朗~
=====================> messages <=====================
final_response: 
content='今天北京天气**晴朗**~ 🌞\n\n适合外出活动，不过建议注意防晒和适时补水。如果需要更详细的温度、风力等信息，可以查看天气预报应用获取实时数据。' additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 86, 'prompt_tokens': 99, 'total_tokens': 185, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': None, 'reasoning_tokens': 45, 'rejected_prediction_tokens': None}, 'prompt_tokens_details': None}, 'model_provider': 'openai', 'mode

In [29]:
from dotenv import load_dotenv
from langchain.chat_models import init_chat_model
import os

load_dotenv(override=True)

MOONSHOT_API_KEY = os.getenv("MOONSHOT_API_KEY")
MOONSHOT_BASE_URL = os.getenv("MOONSHOT_BASE_URL")

model = init_chat_model(
    model="kimi-k2.6",
    model_provider="openai",
    api_key=MOONSHOT_API_KEY,
    base_url=MOONSHOT_BASE_URL
)

from langchain.messages import HumanMessage
@tool
def get_weather(city: str):
 """获取天气的工具"""
 return f"{city}天气晴朗~"

# 将模型和工具绑定
model_with_tools = model.bind_tools([get_weather])

messages = [
HumanMessage("今天北京天气如何")
]

# 模型生成调用工具请求
response = model_with_tools.invoke(messages)

# 添加AIMessage
messages.append(response)
tool_calls = response.tool_calls
for tool_call in tool_calls:
 if tool_call["name"] == "get_weather":
    tool_response = get_weather.invoke(tool_call)
     # 返回的是ToolMessage类型消息
    print(type(tool_response))
    messages.append(tool_response)
print("=====================> messages <=====================")
for msg in messages:
  msg.pretty_print()
print("=====================> messages <=====================")
final_response = model_with_tools.invoke(messages)
print(f"final_response: \n{final_response}")

<class 'langchain_core.messages.tool.ToolMessage'>
=====================> messages <=====================
================================ Human Message =================================

今天北京天气如何
================================== Ai Message ==================================

我来帮您查询今天北京的天气。
Tool Calls:
  get_weather (get_weather_0)
 Call ID: get_weather_0
  Args:
    city: 北京
================================= Tool Message =================================
Name: get_weather

北京天气晴朗~
=====================> messages <=====================
final_response: 
content='今天北京天气**晴朗**~\n\n是个适合出行的好天气！如果您需要更详细的天气信息（如温度、风力等），建议查看专业天气预报平台。\n\n需要查询其他城市的天气吗？' additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 80, 'prompt_tokens': 98, 'total_tokens': 178, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': None, 'reasoning_tokens': 38, 'rejected_prediction_tokens': None}, 'prompt_tokens_details': {'audio_tokens': None, 'cached_toke